# Standardize image filenames

Rewrite image filenames to a canonical schema (`CE<exp>_div<N>_tx<...>_sc<N>_ROI<N>.ome.tif`) so downstream filename parsing in `d00_utils.utilities.extract_img_info` works consistently.

In [ ]:
input_dirpath = Path(input())

In [ ]:
imgnames = [f.name for f in input_dirpath.glob('*.ome.tif')]
imgnames.sort()
df = pd.DataFrame({'prev image name': imgnames})
df.head()

In [ ]:
df.at[0, 'prev image name']

In [ ]:
splits = df['prev image name'].str.replace('-', '_').str.split('.').str[0].str.split('_')
splits[0]

In [ ]:
df['experiment'] = splits.str[0]
df['DIV'] = splits.str[1]
df['wellID'] = splits.str[2] + '-' + splits.str[-2]
df['scene'] = splits.str[-3]
df['new image name'] = df['experiment'] + '_' + df['DIV'] + '_' + df['wellID'] + '_' + df['scene'] + '.ome.tif'

df

In [ ]:
# make sure all new names are unique
assert len(df) == len(df['new image name'].unique())

In [ ]:
tables_dirpath = utils.get_proc_dirpath(input_dirpath) / dn.tables_dirname
tables_dirpath.mkdir(exist_ok=True, parents=True)
df.to_csv(tables_dirpath / 'rename_tifs.csv')

In [ ]:
num_renamed = 0
for i, row in df.iterrows():
    orig_imgpath = input_dirpath / row['prev image name']
    if orig_imgpath.is_file():
        num_renamed = num_renamed + 1
        new_imgpath = input_dirpath / row['new image name']
        orig_imgpath.rename(new_imgpath)
print(num_renamed)